# GRPO (Group Relative Policy Optimization) pada Model Fine-tuned
## Submission Proyek Akhir - PGABL

| **Informasi** | **Detail** |
|---|---|
| **Nama** | Faishal Anwar Hasyim |
| **ID Dicoding** | anwarfaishal86@gmail.com |
| **Kelas** | Pengembangan Generative AI berbasis LLM |

Notebook ini melakukan GRPO menggunakan GRPOTrainer dari TRL dan Unsloth pada model yang telah di-fine-tuning sebelumnya.

---

## 1. Instalasi Library

In [ ]:
%%capture
# Install Unsloth dan dependensi untuk Google Colab
!pip install unsloth[colab-new]
!pip install --no-deps unsloth[colab-no-deps]


## 2. Import Library dan Setup Environment

In [ ]:
import os
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth.chat_templates import get_chat_template

import getpass

# Setup API Keys - kompatibel Colab (browser) & VS Code
def get_secret(key):
    """Ambil secret dari Colab Secrets, env var, atau input manual."""
    try:
        from google.colab import userdata
        return userdata.get(key)
    except Exception:
        val = os.environ.get(key)
        if val:
            return val
        return getpass.getpass(f"Masukkan {key}: ")

os.environ["HF_TOKEN"] = get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = get_secret("WANDB_API_KEY")


print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Masukkan HF_TOKEN: ··········
Masukkan WANDB_API_KEY: ··········
Libraries imported successfully!
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## 3. Memuat dan Mempersiapkan Dataset

Dataset: **Ichsan2895/alpaca-gpt4-indonesian** berisi instruction-output pairs dalam bahasa Indonesia.

In [ ]:
# Memuat dataset dari Hugging Face
dataset = load_dataset("Ichsan2895/alpaca-gpt4-indonesian", split="train")

print(f"Jumlah data: {len(dataset)}")
print(f"Kolom: {dataset.column_names}")
print("\n--- Contoh dataset SEBELUM di-mapping ---")
print(dataset[0])

README.md:   0%|          | 0.00/1.91k [00:00<?, ?B/s]

alpaca-gpt4-indonesia.csv: reconstructing file:   0%|          |  0.00B / 41.4MB            

alpaca-gpt4-indonesia.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/49969 [00:00<?, ? examples/s]

Jumlah data: 49969
Kolom: ['Unnamed: 0', 'input', 'output']

--- Contoh dataset SEBELUM di-mapping ---
{'Unnamed: 0': 1, 'input': 'Saranlah slogan untuk kampanye daur ulang\n', 'output': '1. "Kurangi, gunakan kembali, daur ulang: Bersama untuk masa depan yang lebih hijau."\n2. "Daur ulanglah hari ini, untuk masa depan yang lebih baik."\n3. "Ubah sampahmu menjadi harta karun - Daur ulang!"\n4. "Daur ulang untuk siklus kehidupan."\n5. "Simpan sumber daya, daur ulang lebih banyak."'}


## 4. Memuat Model dengan QLoRA (4-bit, Double Quantization)

Menggunakan **Qwen2.5-1.5B** dari Unsloth.

In [ ]:
max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print(f"\nModel loaded: unsloth/Qwen2.5-1.5B")
print(f"Max sequence length: {max_seq_length}")
print(f"4-bit quantization: {load_in_4bit}")

==((====))==  Unsloth 2026.9.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


Model loaded: unsloth/Qwen2.5-1.5B
Max sequence length: 2048
4-bit quantization: True


## 5. Menambahkan LoRA Adapter

LoRA Adapter diterapkan pada:
- **Multi-Head Attention**: q_proj, k_proj, v_proj, o_proj
- **Feed Forward Network**: gate_proj, up_proj, down_proj

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

print("LoRA Adapter berhasil ditambahkan!")
model.print_trainable_parameters()

Unsloth 2026.9.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


LoRA Adapter berhasil ditambahkan!
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## 6. Mapping Dataset ke Chat Template

Mengubah format dataset Alpaca menjadi format ChatML menggunakan fungsi mapping dari HF datasets.

In [ ]:
tokenizer = get_chat_template(tokenizer, chat_template="chatml")

def formatting_prompts_func(examples):
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for input_text, output_text in zip(inputs, outputs):
        conversation = [
            {"role": "system", "content": "Anda adalah asisten AI yang membantu dan informatif. Jawablah pertanyaan dengan jelas dan akurat dalam Bahasa Indonesia."},
            {"role": "user", "content": input_text.strip()},
            {"role": "assistant", "content": output_text.strip()}
        ]
        text = tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True, desc="Mapping dataset ke Chat Template")

print("\n=== Contoh dataset yang SUDAH di-mapping (dengan token spesial) ===")
print(dataset[0]["text"])

Unsloth: Restored added_tokens_decoder metadata in /content/_unsloth_sentencepiece_temp/tokenizer_lr4fd58a/tokenizer_config.json.


Mapping dataset ke Chat Template:   0%|          | 0/49969 [00:00<?, ? examples/s]


=== Contoh dataset yang SUDAH di-mapping (dengan token spesial) ===
<|im_start|>system
Anda adalah asisten AI yang membantu dan informatif. Jawablah pertanyaan dengan jelas dan akurat dalam Bahasa Indonesia.<|im_end|>
<|im_start|>user
Saranlah slogan untuk kampanye daur ulang<|im_end|>
<|im_start|>assistant
1. "Kurangi, gunakan kembali, daur ulang: Bersama untuk masa depan yang lebih hijau."
2. "Daur ulanglah hari ini, untuk masa depan yang lebih baik."
3. "Ubah sampahmu menjadi harta karun - Daur ulang!"
4. "Daur ulang untuk siklus kehidupan."
5. "Simpan sumber daya, daur ulang lebih banyak."<|im_end|>



## 7. Membagi Dataset menjadi Train dan Validation (Skilled)

In [ ]:
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(eval_dataset)}")

Training samples: 44972
Validation samples: 4997


## 8. Eksperimen Training #1

### Hyperparameter:
- LR: 2e-4, Scheduler: linear, WD: 0.01
- Batch: 2, Grad Accum: 4 (eff. BS: 8)
- Max Steps: 800, Warmup: 50

In [ ]:
import wandb
wandb.login(key=os.environ["WANDB_API_KEY"])

training_args_exp1 = TrainingArguments(
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=50,
    max_steps=800,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="outputs_exp1",
    report_to="wandb",
    run_name="fine-tuning-exp1-lr2e4-wd001",
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

trainer_exp1 = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    dataset_num_proc=2,
    packing=False,
    args=training_args_exp1,
)
print("SFTTrainer Eksperimen 1 siap!")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: catatanfaishal (catatanfaishal-universitas-islam-sultan-agung) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/44972 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/4997 [00:00<?, ? examples/s]

SFTTrainer Eksperimen 1 siap!


In [ ]:
print("=" * 60)
print("MEMULAI TRAINING EKSPERIMEN 1")
print("=" * 60)

# resume_from_checkpoint=True akan otomatis lanjut dari checkpoint terakhir
# jika folder outputs_exp1/checkpoint-* masih ada. Jika tidak, training mulai dari awal.
import os
resume = os.path.isdir("outputs_exp1") and any("checkpoint" in d for d in os.listdir("outputs_exp1"))
if resume:
    print("⏩ Melanjutkan dari checkpoint terakhir...")
else:
    print("▶️ Training dari awal...")

trainer_stats_exp1 = trainer_exp1.train(resume_from_checkpoint=resume)

print("\n" + "=" * 60)
print("TRAINING EKSPERIMEN 1 SELESAI!")
print("=" * 60)
print(f"Training Loss: {trainer_stats_exp1.training_loss:.4f}")
print(f"Total Steps: {trainer_stats_exp1.global_step}")
wandb.finish()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


MEMULAI TRAINING EKSPERIMEN 1
▶️ Training dari awal...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 44,972 | Num Epochs = 1 | Total steps = 800
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
100,1.320617,1.278743
200,1.357218,1.253590
300,1.293727,1.240161
400,1.266845,1.229767
500,1.225539,1.221583
600,1.248082,1.214982
700,1.241759,1.210208
800,1.197546,1.208032


Unsloth: Restored added_tokens_decoder metadata in outputs_exp1/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_exp1/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_exp1/checkpoint-600/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_exp1/checkpoint-800/tokenizer_config.json.



TRAINING EKSPERIMEN 1 SELESAI!
Training Loss: 1.2896
Total Steps: 800


eval/loss,█▆▄▃▂▂▁▁
eval/runtime,▃█▂▂▃▃▂▁
eval/samples_per_second,▆▁▇▇▆▆▇█
eval/steps_per_second,▆▁▇▇▆▆▇█
train/epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇█████
train/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇████
train/grad_norm,▆▆█▄▂▁▃▁▁▂▃▁▁▃▃▃▂▃▂▂▃▃▄▄▄▂▃▂▄▂▂▄▃▄▃▂▃▄▃▃
train/learning_rate,▅███▇▇▇▇▇▇▆▆▆▆▆▆▆▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▁▁▁
train/loss,█▇▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▂▂▂▂▁▂▂▁▂▂▁▂▁▁▁▂▂▂▁▂▂▂▁
eval/loss,1.20803
eval/runtime,512.0466


## 9. Eksperimen Training #2 (Skilled)

### Hyperparameter:
- LR: 5e-5, Scheduler: cosine, WD: 0.05
- Batch: 2, Grad Accum: 8 (eff. BS: 16)
- Max Steps: 800, Warmup: 100

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=3407,
    use_rslora=False, loftq_config=None,
)
tokenizer = get_chat_template(tokenizer, chat_template="chatml")
print("Model re-loaded untuk Eksperimen 2!")
model.print_trainable_parameters()

==((====))==  Unsloth 2026.9.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth: Restored added_tokens_decoder metadata in /content/_unsloth_sentencepiece_temp/tokenizer_u_sph66y/tokenizer_config.json.


Model re-loaded untuk Eksperimen 2!
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [ ]:
training_args_exp2 = TrainingArguments(
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    warmup_steps=100,
    max_steps=800,
    learning_rate=5e-5,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.05,
    lr_scheduler_type="cosine",
    seed=3407,
    output_dir="outputs_exp2",
    report_to="wandb",
    run_name="fine-tuning-exp2-lr5e5-cosine-wd005",
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

trainer_exp2 = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_dataset, eval_dataset=eval_dataset,
    dataset_text_field="text", max_seq_length=max_seq_length,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    dataset_num_proc=2, packing=False, args=training_args_exp2,
)
print("SFTTrainer Eksperimen 2 siap!")

SFTTrainer Eksperimen 2 siap!


In [ ]:
print("=" * 60)
print("MEMULAI TRAINING EKSPERIMEN 2")
print("=" * 60)

resume = os.path.isdir("outputs_exp2") and any("checkpoint" in d for d in os.listdir("outputs_exp2"))
if resume:
    print("⏩ Melanjutkan dari checkpoint terakhir...")
else:
    print("▶️ Training dari awal...")

trainer_stats_exp2 = trainer_exp2.train(resume_from_checkpoint=resume)

print("\n" + "=" * 60)
print("TRAINING EKSPERIMEN 2 SELESAI!")
print("=" * 60)
print(f"Training Loss: {trainer_stats_exp2.training_loss:.4f}")
print(f"Total Steps: {trainer_stats_exp2.global_step}")
wandb.finish()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


MEMULAI TRAINING EKSPERIMEN 2
▶️ Training dari awal...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 44,972 | Num Epochs = 1 | Total steps = 800
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
100,1.375706,1.325373
200,1.290484,1.274992
300,1.305704,1.255302
400,1.224322,1.241937
500,1.257385,1.233622
600,1.198148,1.228651
700,1.232484,1.226510
800,1.277510,1.226140


Unsloth: Restored added_tokens_decoder metadata in outputs_exp2/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_exp2/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_exp2/checkpoint-600/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_exp2/checkpoint-800/tokenizer_config.json.



TRAINING EKSPERIMEN 2 SELESAI!
Training Loss: 1.3390
Total Steps: 800


eval/loss,█▄▃▂▂▁▁▁
eval/runtime,▄▄█▄▂▇▁▂
eval/samples_per_second,▅▅▁▅▇▂█▇
eval/steps_per_second,▅▅▁▅▇▂█▇
train/epoch,▁▁▁▁▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇████
train/global_step,▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇███
train/grad_norm,▆▅██▂▁▁▁▂▂▂▂▂▃▃▂▄▃▅▃▃▄▄▅▄▅▄▅▄▄▄▄▅▄▄▅▄▅▄▄
train/learning_rate,▂▂▅▆▇█████████▇▇▆▆▆▆▅▅▅▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁
train/loss,█▇▆▅▄▂▂▂▂▂▁▂▂▂▁▂▂▂▁▂▁▂▁▂▁▂▂▂▁▁▁▁▁▁▁▁▂▁▂▂
eval/loss,1.22614
eval/runtime,507.0562


## 10. Perbandingan Hasil Eksperimen

In [ ]:
print("=" * 60)
print("PERBANDINGAN HASIL EKSPERIMEN")
print("=" * 60)
print(f"\nEksperimen 1: LR=2e-4, linear, WD=0.01, grad_accum=4")
print(f"  Final Training Loss: {trainer_stats_exp1.training_loss:.4f}")
print(f"\nEksperimen 2: LR=5e-5, cosine, WD=0.05, grad_accum=8")
print(f"  Final Training Loss: {trainer_stats_exp2.training_loss:.4f}")
print("\n=> Pilih eksperimen dengan eval_loss terendah tanpa overfitting.")
print("   Lihat kurva loss di WandB untuk analisis lebih detail.")

PERBANDINGAN HASIL EKSPERIMEN

Eksperimen 1: LR=2e-4, linear, WD=0.01, grad_accum=4
  Final Training Loss: 1.2896

Eksperimen 2: LR=5e-5, cosine, WD=0.05, grad_accum=8
  Final Training Loss: 1.3390

=> Pilih eksperimen dengan eval_loss terendah tanpa overfitting.
   Lihat kurva loss di WandB untuk analisis lebih detail.


## 11. Test Inferensi Model

In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "Anda adalah asisten AI yang membantu dan informatif. Jawablah pertanyaan dengan jelas dan akurat dalam Bahasa Indonesia."},
    {"role": "user", "content": "Apa yang dimaksud dengan hak cipta dalam hukum Indonesia?"}
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=512, use_cache=True, temperature=0.7, top_p=0.9)
response = tokenizer.batch_decode(outputs)
print("=== Test Inferensi ===")
print(response[0])

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Test Inferensi ===
<|im_start|>system
Anda adalah asisten AI yang membantu dan informatif. Jawablah pertanyaan dengan jelas dan akurat dalam Bahasa Indonesia.<|im_end|>
<|im_start|>user
Apa yang dimaksud dengan hak cipta dalam hukum Indonesia?<|im_end|>
<|im_start|>assistant
Hak cipta adalah hak yang diberikan kepada penulis, pengarang, dan penulis lainnya untuk mengembangkan dan menggunakan karya mereka. Ini melibatkan hak untuk mengklaim hak atas karya mereka, hak untuk mengembangkan dan menggunakan karya mereka, dan hak untuk mengembangkan dan menggunakan karya mereka. Hak cipta juga melibatkan hak untuk mengembangkan dan menggunakan karya mereka, dan hak untuk mengembangkan dan menggunakan karya mereka. Hak cipta juga melibatkan hak untuk mengembangkan dan menggunakan karya mereka, dan hak untuk mengembangkan dan menggunakan karya mereka.<|im_end|>


## 12. Upload Model ke Hugging Face

Metode: `merged_16bit`

In [ ]:
HF_USERNAME = get_secret("HF_USERNAME")
MODEL_NAME = f"{HF_USERNAME}/qwen2.5-1.5b-pgabl-legal-sft-faishal"

print(f"Mengunggah model ke: {MODEL_NAME}")
model.push_to_hub_merged(
    MODEL_NAME, tokenizer,
    save_method="merged_16bit",
    token=os.environ["HF_TOKEN"],
)
print(f"\n✅ Model berhasil diunggah ke: https://huggingface.co/{MODEL_NAME}")

Masukkan HF_USERNAME: ··········
Mengunggah model ke: Faishal-Anwar/qwen2.5-1.5b-pgabl-legal-sft-faishal


config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in Faishal-Anwar/qwen2.5-1.5b-pgabl-legal-sft-faishal/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...faishal/model.safetensors:   1%|          | 24.0MB / 3.09GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [02:17<00:00, 137.04s/it]


Unsloth: Merge process complete. Saved to `/content/Faishal-Anwar/qwen2.5-1.5b-pgabl-legal-sft-faishal`

✅ Model berhasil diunggah ke: https://huggingface.co/Faishal-Anwar/qwen2.5-1.5b-pgabl-legal-sft-faishal
